# Per-patient rare populations, disease DEG heatmap and GSEA — real vs PBISC floor (2026-10-31)

**Rare populations**
- Real uses author labels from real halves A + B (sepsis 40 patients; COVID 35 patients, analysed as patient × group).
- Floor uses kNN-transferred labels on the 60k subset (sepsis 121 generated patients; COVID 74, as patient × group).
- The share of each rare population is taken as % of all cells in the unit.
- Statistics per subset and group: SD, binomial-corrected SD, Welch t-test with BH correction, and Levene's test.
- Floor units carry only ~380–500 cells each, so the floor SD contains extra sampling noise. `sd_bio` is the SD after removing the expected binomial variance.

**DEG and GSEA**
- Inputs are the same pseudobulk DEG tables as the deck (`out/allde.pkl`).
- Rank metric: sign(log2FC) × −log10 p.
- gseapy prerank on MSigDB Hallmark and Reactome (2023.2), 1,000 permutations.

In [ ]:
import os, sys; os.chdir('/home/claude/pbisc'); sys.path.insert(0,'/home/claude/pbisc')
import pandas as pd; pd.set_option('display.width',250); pd.set_option('display.max_columns',40)

## 1. Rare populations per patient

In [ ]:
exec(open('perpatient/p1_rare_units.py').read())

In [ ]:
exec(open('perpatient/p1b_disease_shift.py').read())

In [ ]:
exec(open('perpatient/fig_rare_dots.py').read())
from IPython.display import Image, display
for c in ['sepsis','covid']: display(Image(f'perpatient/out/rare_dotplot_{c}.png',width=1100))

## 2. Disease DEG heatmap per cell population

In [ ]:
exec(open('heatgsea/h1_deg_heatmap.py').read())
for c in ['sepsis','covid']: display(Image(f'heatgsea/out/deg_heatmap_{c}.png',width=650))

## 3. GSEA (prerank). Run as `heatgsea/g1_gsea.py`, which takes about 10 min:
```python
"""Preranked GSEA of the disease contrast per cell population, real vs PBISC floor (same pseudobulk DEG tables).
Rank = sign(log2FC) * -log10(p) (Welch). Gene sets: MSigDB Hallmark 2023.2 and Reactome 2023.2, restricted to the ~5,700-gene panel.
gseapy prerank, 1,000 gene-set permutations, size 10-500 in panel, seed 0."""
import numpy as np, pandas as pd, pickle, gseapy as gp, os, sys, warnings; warnings.filterwarnings("ignore")
A=pickle.load(open("out/allde.pkl","rb")); SIX=["T cell","B cell","NK cell","Monocyte","DC"]
GS={"hallmark":"heatgsea/../gsea/genesets/h.all.v2023.2.Hs.symbols.gmt","reactome":"gsea/genesets/c2.cp.reactome.gmt"}
out=[]
for lib,path in GS.items():
    for coh in ["sepsis","covid"]:
        for ct in SIX:
            for src in ["real","floor"]:
                t=A[(coh,ct)][src].dropna(subset=["p"]); p=np.clip(t.p.values,1e-300,1)
                rnk=pd.Series(np.sign(t.logFC.values)*-np.log10(p),index=t.gene.values).sort_values(ascending=False)
                rnk=rnk[~rnk.index.duplicated()]+np.random.default_rng(0).normal(0,1e-6,len(rnk))   # break ties
                res=gp.prerank(rnk=rnk,gene_sets=path,min_size=10,max_size=500,permutation_num=1000,seed=0,threads=2,outdir=None,verbose=False)
                r=res.res2d[["Term","ES","NES","NOM p-val","FDR q-val","Tag %","Lead_genes"]].copy()
                r.columns=["term","ES","NES","p","fdr","tag","lead_genes"]; r["library"]=lib; r["cohort"]=coh; r["celltype"]=ct; r["source"]=src
                out.append(r); print(lib,coh,ct,src,len(r),flush=True)
G=pd.concat(out,ignore_index=True)
for c in ["ES","NES","p","fdr"]: G[c]=pd.to_numeric(G[c],errors="coerce")
G.to_csv("heatgsea/out/gsea_all.csv.gz",index=False,compression="gzip")

```

In [ ]:
exec(open('heatgsea/g2_compare.py').read())
for f in ['gsea_hallmark_heatmap_sepsis','gsea_hallmark_heatmap_covid','gsea_nes_scatter']: display(Image(f'heatgsea/out/{f}.png',width=800))

In [ ]:
import pickle, numpy as np
A=pickle.load(open('out/allde.pkl','rb'))
for coh in ['sepsis','covid']:
    for ct in ['T cell','B cell','NK cell','Monocyte','DC']:
        print(coh,ct,' | '.join(f"{s}: RP genes median log2FC {A[(coh,ct)][s][A[(coh,ct)][s].gene.str.match('^RP[SL][0-9]')].logFC.median():+.2f}" for s in ['real','floor']))